In [1]:
import openai
import os

openai.__version__

'1.76.2'

In [29]:
miracl_n_hard_negs = 1000
miracl_n_recalls = [1,3,10,30,100,300,1000]
miracl_n_corpus = 2000
miracl_n_queries = 100
model_id = ""
dimension = -1
query_prefix = ""
passage_prefix = ""

In [30]:
# Parameters
model_id = "text-embedding-3-small"
dimension = 1536

In [31]:
# cache
tmpdir = f"tmp/{model_id}_{dimension}"
os.makedirs(tmpdir, exist_ok=True)


# Model

In [34]:
import numpy as np
import dotenv
from langchain_openai import OpenAIEmbeddings

dotenv.load_dotenv("openai_key", override=True)

if "text-embedding-3" not in model_id:
    client = OpenAIEmbeddings(model=model_id)
else:
    client = OpenAIEmbeddings(model=model_id, dimensions=dimension)

def get_embeddings(texts: list[str]) -> np.ndarray:
    texts = [text.replace("\n", " ")[:2000] for text in texts]
    all_embeddings = []
    for i in range(0, len(texts), 1000):
        embs = client.embed_documents(texts[i : i + 1000])
        all_embeddings += embs
    return np.array(all_embeddings)


# Miracl
* Need access token for huggingface

In [6]:
import os
import json

dotenv.load_dotenv("huggingface_access_token", override=True)

True

In [9]:
import datasets

# query and positives
ds = datasets.load_dataset(
    "miracl/miracl", "ja", token=os.environ["HF_ACCESS_TOKEN"], split="dev"
)
ds

Dataset({
    features: ['query_id', 'query', 'positive_passages', 'negative_passages'],
    num_rows: 860
})

In [10]:
# all corpus texts
corpus = datasets.load_dataset("miracl/miracl-corpus", "ja")
corpus

DatasetDict({
    train: Dataset({
        features: ['docid', 'title', 'text'],
        num_rows: 6953614
    })
})

In [11]:
# hard negatives
with open("./miracl_hard_negs_1000.json") as f:
    hn = json.loads(f.read())
len(hn), list(hn.keys())[:5], hn["0"].keys(), hn["0"]["docids"][:2], hn["0"]["indices"][
    :2
]

(860,
 ['0', '3', '4', '5', '7'],
 dict_keys(['docids', 'indices']),
 ['2681119#0', '2681119#1'],
 [1393435, 1393436])

In [13]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist


def get_text(corpus_item):
    return corpus_item["title"] + " " + corpus_item["text"]


corpus_dict = {item["docid"]: get_text(item) for item in corpus["train"]}

In [ ]:
# only evaluate first 100 queries
ds_subset = ds.select(range(miracl_n_queries))
# all positive and hard neg docids
all_positive_docids = set()
all_hard_neg_docids = set()
for item in ds_subset:
    all_positive_docids.update([pp["docid"] for pp in item["positive_passages"]])
    all_hard_neg_docids.update(hn[item["query_id"]]["docids"][:miracl_n_hard_negs])
# positiveもhard_negも含まないmiracl_n_corpusくらいのサブセットを作る
miracl_n_initial_subset = miracl_n_corpus + len(all_positive_docids) + miracl_n_hard_negs*miracl_n_queries
corpus_initial_subset = corpus["train"].select(range(miracl_n_initial_subset))

corpus_subset = corpus_initial_subset.filter(lambda x: x["docid"] not in all_positive_docids and x["docid"] not in all_hard_neg_docids)
corpus_subset = corpus_subset.select(range(miracl_n_corpus - miracl_n_hard_negs))
# コーパスサブセットのサイズをprint
if len(corpus_subset) < miracl_n_corpus - miracl_n_hard_negs:
    print(f"コーパスサブセットのサイズ {len(corpus_subset)}は{miracl_n_corpus - miracl_n_hard_negs}より小さいです。")
else:
    print(f"コーパスサブセットのサイズは{miracl_n_corpus - miracl_n_hard_negs}です。")


Filter: 100%|██████████| 102195/102195 [00:00<00:00, 177818.97 examples/s]

コーパスサブセットのサイズは1000です。


In [26]:
# corpus_subsetのembeddingは繰り返し使うので先に計算しておく
tmppath = f'{tmpdir}/corpus_subset.npy'
if os.path.exists(tmppath):
    corpus_subset_embs = np.load(tmppath)
else:
    corpus_subset_embs = get_embeddings([passage_prefix + get_text(item) for item in corpus_subset])
    np.save(tmppath, corpus_subset_embs)


In [35]:
# queryのサブセットのembeddingもまとめて取得しておく
tmppath = f'{tmpdir}/query_subset.npy'
if os.path.exists(tmppath):
    query_subset_embs = np.load(tmppath)
else:
    query_subset_embs = get_embeddings([query_prefix + item["query"] for item in ds_subset])
    np.save(tmppath, query_subset_embs)


In [36]:
import tqdm

n_total_pos = 0
n_total_tps = [0] * len(miracl_n_recalls)

# only evaluate first 100 queries
for item, query_emb in tqdm.tqdm(zip(ds_subset, query_subset_embs)):

    # passages are set(300 hard negatives + positives)
    positive_docids = [pp["docid"] for pp in item["positive_passages"]]
    positive_texts = [get_text(pp) for pp in item["positive_passages"]]
    hn_docids = hn[item["query_id"]]["docids"][:miracl_n_hard_negs]

    # drop hard negatives in positives
    hn_docids = [docid for docid in hn_docids if docid not in positive_docids]

    # search target
    target_docids = positive_docids + hn_docids
    target_texts = positive_texts + [corpus_dict[docid] for docid in hn_docids]

    # embedding
    tmppath = f'{tmpdir}/{item["query_id"]}.npy'
    if os.path.exists(tmppath):
        target_embs = np.load(tmppath)
    else:
        # use cache
        target_embs = get_embeddings([passage_prefix + text for text in target_texts])
        np.save(tmppath, target_embs)

    # target_embsとcorpus_subset_embsを結合
    target_embs = np.concatenate([target_embs, corpus_subset_embs], axis=0)
    sorted_indices = np.argsort(cdist(query_emb, target_embs, metric="cosine"))[0]

    n_pos = len(positive_docids)
    n_tps = []
    for miracl_n_recall in miracl_n_recalls:
        topk_indices = sorted_indices[:miracl_n_recall]
        n_tp = len(
            set(topk_indices) & set(range(len(positive_docids)))
        )  # positives are first indices
        n_tps.append(n_tp)
        

    n_total_pos += n_pos
    for i in range(len(miracl_n_recalls)):
        n_total_tps[i] += n_tps[i]

miracl_recalls = [n_total_tps[i] / n_total_pos for i in range(len(miracl_n_recalls))]


print(n_total_pos)
for i in range(len(miracl_n_recalls)):
    print(miracl_n_recalls[i], n_total_tps[i], miracl_recalls[i])

0it [00:00, ?it/s]


ValueError: XA must be a 2-dimensional array.

# Output

In [19]:
model_id, dimension, miracl_recalls

('text-embedding-3-small',
 1536,
 0.7809981604825089,
 0.8040307567839314,
 0.7948717948717948)

In [20]:
import json

with open(f'./scores/{model_id.replace("/", "_")}_{dimension}.txt', "w") as f:
    f.write(
        json.dumps(
            {
                "model_id": model_id,
                "miracl_n_recalls": miracl_n_recalls,
                "miracl_recalls": miracl_recalls
            }
        )
    )